# Keithley 2460 hardware timestamp proof

This guarded notebook measures the relationship between a physical external START edge, the BUSY output, the instrument event-log clock, and the first buffered reading timestamp. It is **only** for the front-terminal four-wire nominal 100 Ω resistor setup. Every mutating or consuming action is disabled by default.

Use a scope with high-impedance inputs: CH1 on START (digital line 3), CH2 on BUSY (digital line 2), and both grounds on digital-I/O ground (DB-9 pin 9). Capture the rising START-to-rising BUSY delay and its uncertainty. The proof uses rising-edge START, +1 mA, a 0.2 V limit, and one 1 NPLC reading. Do not connect a battery.

In [ ]:
from __future__ import annotations

import hashlib
import json
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path

from ophyd_electrochemistry.acquisition import StartMode
from ophyd_electrochemistry.keithley.k2460 import (
    BufferPolicy,
    CurrentHoldAcquisitionProof,
    DigitalIOConfig,
    Keithley2460Config,
    M4RuntimeController,
    PyVisaKeithley2460Transport,
    RuntimeState,
    SafetyConfig,
    TimingPolicy,
    VisaTransportConfig,
    analyze_timestamp_proof,
    packaged_runtime,
)
from ophyd_electrochemistry.keithley.k2460.commissioning import run_read_only_diagnostic

RESOURCE = "TCPIP0::169.254.113.151::5025::SOCKET"
PROOF_RUN_ID = 2026101001  # Change for every attempt.
EVIDENCE_DESTINATION = None  # Example: ../docs/evidence/k2460-YYYY-MM-DD-timestamp-proof.json
SCOPE_START_TO_BUSY_S = None  # Enter the measured rising-edge delay before retrieval.
SCOPE_UNCERTAINTY_S = None  # Enter a conservative absolute uncertainty.

RUN_QUERY_ONLY_PREFLIGHT = False
INSTALL_OR_REPLACE_VOLATILE_RUNTIME = False
PREPARE_AND_ARM_TIMESTAMP_PROOF = False
RETRIEVE_AND_CONSUME_INFO_EVENTS = False
RUN_INDEPENDENT_AUDIT = False
DISCARD_RETAINED_BUFFER = False
CONFIRM_FRONT_4WIRE_100_OHM_AND_OUTPUT_OFF = False
CONFIRM_SCOPE_ON_START_AND_BUSY = False
CONFIRM_GENERATOR_LOW_BEFORE_ARM = False

In [ ]:
artifact = packaged_runtime()
assert artifact.build == "m4-finite-current-hold-v13"
assert artifact.script_name == f"oe_m4_{artifact.sha256[:24]}"
assert "trigger.BLOCK_LOG_EVENT" in artifact.source
assert "oe_m5_timestamp_proof_reading" in artifact.source
print({"abi": artifact.abi, "build": artifact.build, "sha256": artifact.sha256})

In [ ]:
if not RUN_QUERY_ONLY_PREFLIGHT:
    raise RuntimeError("STOP: query-only hardware preflight is disabled by default.")

visa_config = VisaTransportConfig(
    resource_name=RESOURCE, backend="@py", required_language="TSP", io_timeout_ms=5_000
)
preflight = run_read_only_diagnostic(visa_config)
assert preflight.output_enabled is False
print(json.dumps(preflight.as_dict(), indent=2, sort_keys=True))

In [ ]:
config = Keithley2460Config(
    visa_resource=RESOURCE,
    source_terminal="front",
    sense="remote",
    io=DigitalIOConfig(start_edge="rising"),
    buffer=BufferPolicy(capacity_ceiling_records=16, transfer_chunk_records=16),
    safety=SafetyConfig(
        voltage_min_v=-0.2,
        voltage_max_v=0.2,
        charge_current_max_a=0.001,
        discharge_current_max_a=0.001,
        power_abs_max_w=0.001,
        source_off_mode="normal",
    ),
    timing=TimingPolicy(
        command_timeout_s=10,
        external_start_timeout_s=60,
        shutdown_timeout_s=2,
        poll_period_s=0.01,
        required_abort_latency_s=1,
        required_cutoff_latency_s=1,
    ),
)
request = CurrentHoldAcquisitionProof(
    current_a=0.001,
    source_range_a=0.001,
    voltage_limit_v=0.2,
    measurement_count=1,
    start_mode=StartMode.EXTERNAL_TRIGGER,
)


def new_transport(io_timeout_ms: int = 10_000) -> PyVisaKeithley2460Transport:
    return PyVisaKeithley2460Transport(
        VisaTransportConfig(
            resource_name=RESOURCE,
            backend="@py",
            required_language="TSP",
            io_timeout_ms=io_timeout_ms,
            script_line_delay_s=0.01,
        )
    )

In [ ]:
if not CONFIRM_FRONT_4WIRE_100_OHM_AND_OUTPUT_OFF:
    raise RuntimeError("STOP: resistor wiring and physical output-off state are not confirmed.")
if not INSTALL_OR_REPLACE_VOLATILE_RUNTIME:
    raise RuntimeError("STOP: volatile runtime installation is disabled by default.")

transport = new_transport()
try:
    transport.connect()
    controller = M4RuntimeController(transport, config)
    loaded_build = transport.query("print(oe_m4_runtime_build)")
    if loaded_build == artifact.build:
        assert transport.query("print(oe_m4_runtime_abi)") == artifact.abi
        assert transport.query(f"print({artifact.script_name} == nil)") == "false"
        installed = artifact
    else:
        installed = controller.replace_known_runtime_and_install()
    status = controller.status()
    assert status.state == RuntimeState.IDLE and not status.output_enabled
    assert float(transport.query("print(smu.source.level)")) == 0.0
    assert controller.buffer_info().record_count == 0
    print(
        json.dumps({"runtime": asdict(installed), "status": asdict(status)}, default=str, indent=2)
    )
finally:
    transport.close()

## Scope and trigger procedure

Before arming, hold START low and verify both scope channels. Configure a single rising-edge capture on START. Run the next cell; it must report `WAITING_START`, READY high, BUSY low, and output off. Then produce exactly one rising START edge. Record the delay from the START threshold crossing to the BUSY rising edge and a conservative uncertainty. Keep START high until the run completes; its later falling edge is ignored because the configured trigger is rising-only.

In [ ]:
if not all(
    (
        CONFIRM_FRONT_4WIRE_100_OHM_AND_OUTPUT_OFF,
        CONFIRM_SCOPE_ON_START_AND_BUSY,
        CONFIRM_GENERATOR_LOW_BEFORE_ARM,
    )
):
    raise RuntimeError("STOP: all physical confirmations are required before arm.")
if not PREPARE_AND_ARM_TIMESTAMP_PROOF:
    raise RuntimeError("STOP: timestamp proof arm is disabled by default.")

transport = new_transport()
try:
    transport.connect()
    controller = M4RuntimeController(transport, config)
    assert transport.query("print(oe_m4_runtime_build)") == artifact.build
    initial = controller.status()
    assert initial.state == RuntimeState.IDLE and not initial.output_enabled
    assert float(transport.query("print(smu.source.level)")) == 0.0
    showevents_mask = controller.showevents_mask()
    assert showevents_mask & 4 == 0, (
        "Disable automatic display of information events before this proof."
    )
    event_counts_before = {
        "information": controller.information_event_count(),
        "warnings": int(transport.query("print(eventlog.getcount(eventlog.SEV_WARN))")),
        "errors": int(transport.query("print(eventlog.getcount(eventlog.SEV_ERROR))")),
    }
    prepared = controller.prepare_timestamp_proof(request, proof_run_id=PROOF_RUN_ID)
    armed = controller.arm()
    assert armed.state == RuntimeState.WAITING_START
    assert not armed.output_enabled and armed.ready_level == 1 and armed.busy_level == 0
    print("ARMED: capture and produce exactly one rising START edge now.", flush=True)
    print(json.dumps(asdict(armed), default=str, indent=2))
finally:
    transport.close()

In [ ]:
if not RETRIEVE_AND_CONSUME_INFO_EVENTS:
    raise RuntimeError(
        "STOP: event retrieval is disabled; eventlog.next consumes remote-unread entries."
    )
if SCOPE_START_TO_BUSY_S is None or SCOPE_UNCERTAINTY_S is None:
    raise RuntimeError("STOP: enter the scope delay and conservative uncertainty first.")
if EVIDENCE_DESTINATION is None:
    raise RuntimeError(
        "STOP: choose a new evidence destination; existing files are never overwritten."
    )

transport = new_transport()
controller = None
try:
    transport.connect()
    controller = M4RuntimeController(transport, config)
    terminal = controller.status()
    assert terminal.state == RuntimeState.COMPLETE and not terminal.output_enabled
    reading = controller.timestamp_proof_reading()
    raw_chunk = controller.read_buffer_chunk(offset=0, max_records=1)
    assert len(raw_chunk.records) == 1
    info_count = controller.information_event_count()
    assert 4 <= info_count <= 100, "Refuse an unbounded event-log drain."
    events = tuple(controller.next_information_event() for _ in range(info_count))
    analysis = analyze_timestamp_proof(
        reading,
        events,
        proof_run_id=PROOF_RUN_ID,
        start_to_busy_s=SCOPE_START_TO_BUSY_S,
        scope_uncertainty_s=SCOPE_UNCERTAINTY_S,
    )
    counts_after = {
        "information_remote_unread": controller.information_event_count(),
        "warnings": int(transport.query("print(eventlog.getcount(eventlog.SEV_WARN))")),
        "errors": int(transport.query("print(eventlog.getcount(eventlog.SEV_ERROR))")),
    }
    assert counts_after["warnings"] == event_counts_before["warnings"]
    assert counts_after["errors"] == event_counts_before["errors"]
    recovered = controller.recover()
    assert recovered.state == RuntimeState.IDLE and not recovered.output_enabled
    assert transport.query("print(smu.source.output)") == "smu.OFF"
    assert float(transport.query("print(smu.source.level)")) == 0.0
    evidence = {
        "schema": "ophyd-electrochemistry/k2460-timestamp-proof-v1",
        "recorded_at_utc": datetime.now(UTC).isoformat().replace("+00:00", "Z"),
        "instrument_identity": transport.identity.raw,
        "resource": RESOURCE,
        "runtime": {
            "abi": artifact.abi,
            "build": artifact.build,
            "sha256": artifact.sha256,
            "script_name": artifact.script_name,
        },
        "proof_run_id": PROOF_RUN_ID,
        "request": asdict(request),
        "showevents_mask": showevents_mask,
        "event_counts_before": event_counts_before,
        "event_counts_after": counts_after,
        "terminal": asdict(terminal),
        "reading": asdict(reading),
        "raw_record": asdict(raw_chunk.records[0]),
        "events_consumed": [asdict(event) for event in events],
        "scope": {"start_to_busy_s": SCOPE_START_TO_BUSY_S, "uncertainty_s": SCOPE_UNCERTAINTY_S},
        "analysis": asdict(analysis),
        "assessment": "UNREVIEWED: no timestamp-reference or "
        + "hardware gate claim is made automatically",
        "manual": {
            "part": "2460-901-01C",
            "revision": "C / September 2019",
            "url": "https://download.tek.com/manual/2460-901-01C_Sept_2019_Ref.pdf",
        },
    }
    canonical = json.dumps(evidence, default=str, sort_keys=True, separators=(",", ":"))
    evidence["sha256"] = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
    destination = Path(EVIDENCE_DESTINATION)
    with destination.open("x", encoding="utf-8") as stream:
        json.dump(evidence, stream, default=str, indent=2, sort_keys=True)
        stream.write("\n")
    print(json.dumps(evidence, default=str, indent=2, sort_keys=True))
except BaseException:
    if controller is not None and transport.connected:
        try:
            controller.force_safe()
        except BaseException:
            pass
    raise
finally:
    transport.close()

In [ ]:
if not RUN_INDEPENDENT_AUDIT:
    raise RuntimeError("STOP: independent audit is disabled by default.")
transport = new_transport(io_timeout_ms=5_000)
try:
    transport.connect()
    controller = M4RuntimeController(transport, config)
    status = controller.status()
    audit = {
        "status": asdict(status),
        "output": transport.query("print(smu.source.output)"),
        "source_level_a": float(transport.query("print(smu.source.level)")),
        "buffer": asdict(controller.buffer_info()),
    }
    assert status.state == RuntimeState.IDLE and not status.output_enabled
    assert audit["output"] == "smu.OFF" and audit["source_level_a"] == 0.0
    print(json.dumps(audit, default=str, indent=2, sort_keys=True))
finally:
    transport.close()

Only discard after the evidence file and independent audit have been checked. This destroys the retained reading and cannot be undone.

In [ ]:
if not DISCARD_RETAINED_BUFFER:
    raise RuntimeError("STOP: destructive retained-buffer discard is disabled by default.")
transport = new_transport(io_timeout_ms=5_000)
try:
    transport.connect()
    discarded = M4RuntimeController(transport, config).discard_buffer(
        reason="timestamp proof archived and independently audited"
    )
    assert discarded.record_count == 0
    print(asdict(discarded))
finally:
    transport.close()